# Macrophage annotation

Takes the macrophage object out of `immune_split.ipynb` and assigns `cell_type`
labels. Despite the name this object holds the whole myeloid lineage -
macrophages, monocytes and dendritic cells - which is how it came out of the
three-way immune split.

Two clustering passes: cluster and drop what does not belong, then re-cluster
what remains and annotate it. Removing cells changes every neighbourhood, so
the graph and UMAP are rebuilt in between.

Set `DATA` in `config.py` before running.

In [ ]:
import pandas as pd
import scanpy as sc

from config import DATA

SEED = 0

## 1. Load the macrophage object

Written by `immune_split.ipynb`.

In [ ]:
adata = sc.read_h5ad(DATA / 'macrophages.h5ad')
print(adata.shape)

## 2. First pass - cluster to find the junk

**The resolution below is an example, not a setting to copy.** Tune it by eye
until the cells you want to remove sit in a cluster of their own, and check the
result yourself.

In [ ]:
sc.pp.neighbors(adata, use_rep='X_scVI', n_neighbors=20, random_state=SEED)
sc.tl.umap(adata, random_state=SEED)

# example - tune by eye until the junk separates
sc.tl.leiden(adata, resolution=0.4, random_state=SEED)

print(f"{adata.obs['leiden'].nunique()} clusters")
sc.pl.umap(adata, color='leiden', palette='tab20', legend_loc='on data',
           frameon=False)

In [ ]:
sc.pl.umap(adata, color='n_genes_by_counts', frameon=False)

## 3. Drop the junk clusters

**The list below is empty on purpose.** Which clusters come out depends on the
resolution above, so read them off the plots and fill it in.

In [ ]:
JUNK = []   # read off the plots above

n_before = adata.n_obs
adata = adata[~adata.obs['leiden'].isin(JUNK)].copy()

print(f'{n_before:,} -> {adata.n_obs:,} cells ({n_before - adata.n_obs:,} dropped)')

## 4. Second pass - re-cluster what is left

In [ ]:
sc.pp.neighbors(adata, use_rep='X_scVI', n_neighbors=20, random_state=SEED)
sc.tl.umap(adata, random_state=SEED)

# example - tune by eye until the populations separate
sc.tl.leiden(adata, resolution=0.4, random_state=SEED)

print(f"{adata.obs['leiden'].nunique()} clusters")
sc.pl.umap(adata, color='leiden', palette='tab20', legend_loc='on data',
           frameon=False)

## 5. Markers

One panel across the whole lineage, covering resident macrophages, the
monocyte-derived populations, the dendritic cell subsets and the cycling
fraction.

In [ ]:
MARKERS = ['Marco', 'Spp1', 'Cd74', 'Ly6c2', 'Ccl2', 'Mki67', 'Ccr2',
           'Cd300a', 'Ccr7', 'Atox1', 'Fn1', 'Arg1', 'Cst3', 'Plbd1',
           'Xbp1', 'Bst2']

sc.pl.umap(adata, color=MARKERS, frameon=False)

Ranked marker genes per cluster, as a cross-check on the panel above.

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden', method='t-test')

# one column per cluster, top 100 genes down the rows - this is what the calls
# below were read from
top_genes = pd.DataFrame(adata.uns['rank_genes_groups']['names']).head(100)
top_genes.head(30)

## 6. Assign cell types

**The dict below is an example.** The cluster numbers, and how many there are,
follow from whatever resolution was used above - so this mapping has to be
rebuilt rather than reused. Read the marker panel and the ranked genes, and
check every cluster yourself.

Several clusters carrying the same label is normal: Leiden splits the large
populations further than the biology justifies, and merging them back is the
point.

In [ ]:
# example only - rebuild this for the clustering you get
CELL_TYPES = {
    '0':  'Interstitial Macrophages',
    '1':  'Alveolar Macrophages',
    '2':  'Spp1+ Alveolar Macrophages (MoAMs)',
    '3':  'cDC1',
    '4':  'Classical Monocytes',
    '5':  'Spp1+ Interstitial Macrophages',
    '6':  'Interstitial Macrophages',
    '7':  'pDC1',
    '8':  'Activated Monocytes',
    '9':  'Cycling Myeloid Cells',
    '10': 'cDC2',
    '11': 'Spp1+ Alveolar Macrophages (MoAMs)',
    '12': 'pDC2',
}

adata.obs['cell_type'] = adata.obs['leiden'].map(CELL_TYPES).astype('category')

print(adata.obs['cell_type'].value_counts().to_string())

In [ ]:
sc.pl.umap(adata, color='cell_type', frameon=False)

Composition per condition - the reason the annotation exists.

In [ ]:
sc.pl.umap(adata, color=['type', 'age'], frameon=False)

## 7. Save

In [ ]:
adata.write_h5ad(DATA / 'macrophages_annotated.h5ad')
print(f'{adata.n_obs:,} cells, {adata.obs["cell_type"].nunique()} cell types')